# 01 · La comparación ingenua

**Objetivo:** Medir cuánto de una diferencia de precio entre municipios se mantiene cuando se comparan productos, presentaciones y periodos equivalentes, y distinguir lo que los datos públicos permiten afirmar de lo que haría falta para evaluar una oportunidad de negocio.

**Pregunta:** Si se comparan precios entre municipios a lo bruto (por nombre de producto), ¿qué diferencias aparecen?

Por qué importa: es la cifra que aparece cuando alguien abre el archivo de PROFECO, agrupa por nombre de
producto y compara el promedio de cada municipio. Es el punto de partida del embudo y el contraste para todo lo demás.

**Correlación de rangos (Spearman)** = qué tanto se ordenan igual dos variables: 1 es el mismo orden, 0 ninguna relación.

**Decisión:** la diferencia entre municipios de un nombre de producto es la brecha P90/P10 de sus promedios
municipales (cuánto más cuesta en el municipio caro típico que en el barato típico). El universo son los 280 nombres
de producto que llegan validados al final del embudo (sin medicamentos), para comparar siempre lo mismo.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
raw = entorno.load_raw()
d_all = M.prep(raw)
d = d_all[~d_all.catalogo.isin(M.FUERA_PRINCIPAL)].copy()      # universo principal
params = json.loads((META / "parametros.json").read_text(encoding="utf-8"))
REPS = params.get("bootstrap_replicas", 500)
C.register_cities(d_all.municipio_id.unique())
res, prod, un, pm = M.embudo(d)
U = set(prod.index)
bt = M.boot_embudo({"ingenua": pm["ingenua"]}, {"ingenua": ["producto"]}, U, reps=REPS)
lo, hi = np.percentile(bt["ingenua"], [5, 95])
F = "01_comparacion_ingenua"
reg("ingenua", prod.ingenua.median(), (lo, hi), n=len(prod), fuente=F, nota="brecha P90/P10 mediana entre nombres de producto, comparación ingenua")
print(f"Comparación ingenua: {prod.ingenua.median():.1f}% (IC90 {lo:.1f}–{hi:.1f}) en {len(prod)} nombres de producto")
display(prod.ingenua.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_frame("brecha ingenua (%)").T)

Comparación ingenua: 25.0% (IC90 23.3–28.2) en 279 nombres de producto


,count,mean,std,min,10%,25%,50%,75%,90%,max
brecha ingenua (%),279.0,43.7,65.6,3.3,10.4,15.7,25.0,42.4,82.9,537.9


## 1. ¿Cuántas diferencias «parecen» oportunidad?

Umbrales de lectura: una diferencia de 30% o más entre el municipio caro típico y el barato típico es la que suele
llamar la atención en una presentación comercial.

In [2]:
umbr = pd.DataFrame({"umbral": [10, 20, 30, 50, 100]})
umbr["% de nombres por encima (ingenua)"] = [(prod.ingenua > u).mean() * 100 for u in umbr.umbral]
display(umbr.round(1))
reg("pct_ingenua_30", (prod.ingenua > 30).mean() * 100, n=len(prod), fuente=F, nota="% de nombres con brecha ingenua > 30%")
reg("pct_ingenua_50", (prod.ingenua > 50).mean() * 100, n=len(prod), fuente=F)
reg("ingenua_promedio", prod.ingenua.mean(), n=len(prod), fuente=F)

,umbral,% de nombres por encima (ingenua)
0,10,91.8
1,20,64.9
2,30,43.4
3,50,20.4
4,100,7.9


## 2. Las diferencias ingenuas más grandes

Para cada nombre: cuántos artículos distintos mezcla (presentación × marca) y el promedio en el municipio barato y el
caro típicos (P10 y P90).

In [3]:
mix = d.groupby("producto").agg(articulos=("sku_id", "nunique"), presentaciones=("pres_key", "nunique"),
                                 marcas=("marca", "nunique"), giros=("giro", "nunique"))
ing = un["ingenua"].set_index("producto").loc[sorted(U)].join(mix)
top = ing.sort_values("brecha_pct", ascending=False).head(12)
display(top[["municipios", "p10", "p90", "brecha_pct", "articulos", "presentaciones", "marcas", "giros"]].round(1))
rho = ing[["brecha_pct", "articulos", "presentaciones", "marcas"]].corr("spearman").brecha_pct
display(rho.round(2).to_frame("correlación de rangos con la brecha ingenua"))
reg("rho_articulos", rho["articulos"], n=len(ing), fuente=F, nota="Spearman entre brecha ingenua y número de artículos del nombre")
ing.reset_index().to_csv(PROC / "brecha_ingenua.csv", index=False, encoding="utf-8-sig")

,municipios,p10,p90,brecha_pct,articulos,presentaciones,marcas,giros
producto,,,,,,,,
Almendras,60,36.5,232.8,537.9,4,3,4,5
Nuez,56,57.9,345.2,496.3,4,3,3,5
Chile Seco,57,38.5,226.9,489.4,8,3,3,4
Canela,62,41.1,209.8,410.5,6,5,5,4
Folders,49,32.8,147.8,350.5,8,8,1,2
Amaranto,52,30.0,96.0,220.0,2,2,2,3
Ajonjolí,54,21.0,64.2,205.7,4,3,4,4
Extractores de Jugos y Exprimidores,61,725.0,1989.3,174.4,15,15,8,3
Queso Fresco,59,63.7,160.7,152.4,2,2,2,1


,correlación de rangos con la brecha ingenua
brecha_pct,1.00
articulos,0.11
presentaciones,0.11
marcas,-0.03


## 3. Con todo el archivo

Como contexto, la misma comparación ingenua con todos los nombres de producto que están en 10 municipios o más,
incluidos los medicamentos y los que no llegan al final del embudo.

In [4]:
todo = M.brechas(M.precios_municipales(d_all, "ingenua"), ["producto"])
reg("ingenua_todos_nombres", todo.brecha_pct.median(), n=len(todo), fuente=F)
print(f"Todos los nombres con 10+ municipios: {len(todo)} · brecha ingenua mediana {todo.brecha_pct.median():.1f}%")
v = entorno.cifras()
display(Markdown(f'''**Conclusión.** A lo bruto, el producto típico cuesta **{v["ingenua"]:.0f}% más** en el municipio caro
típico que en el barato típico (IC90 {v["ingenua_inf"]:.0f}–{v["ingenua_sup"]:.0f}%), y {v["pct_ingenua_30"]:.0f}% de los
nombres de producto superan 30%. El tamaño de esa diferencia casi no depende de cuántos artículos mezcle el nombre
(correlación de rangos {v["rho_articulos"]:.2f}): depende de qué presentación o marca domina en cada municipio, que es lo
que el embudo corrige.'''))

Todos los nombres con 10+ municipios: 737 · brecha ingenua mediana 24.8%


**Conclusión.** A lo bruto, el producto típico cuesta **25% más** en el municipio caro
típico que en el barato típico (IC90 23–28%), y 43% de los
nombres de producto superan 30%. El tamaño de esa diferencia casi no depende de cuántos artículos mezcle el nombre
(correlación de rangos 0.11): depende de qué presentación o marca domina en cada municipio, que es lo
que el embudo corrige.